# AuditorAgent Fine-Tuning Pipeline — Qwen2.5-7B-Instruct
## QLoRA Training of Qwen/Qwen2.5-7B-Instruct for Architectural Governance Auditing

**Task:** Train a specialised AuditorAgent that produces structured JSON audit reports  
**Method:** QLoRA (4-bit NormalFloat quantisation) with completion-only loss masking  
**Base Model:** Qwen/Qwen2.5-7B-Instruct (7B parameters)

---

### Key Design Decisions vs Phi-3 baseline

| Decision | Phi-3-mini (baseline) | Qwen2.5-7B |
|----------|----------------------|-------------|
| Parameters | 3.8B | 7B |
| 4-bit memory | ~2GB | ~4GB |
| Response template | `<\|assistant\|>` | `<\|im_start\|>assistant\n` |
| End token | `<\|end\|>` | `<\|im_end\|>` |
| trust_remote_code | True | False |
| HF token | No | No |
| Chat format | Phi-3 native | ChatML format |
| JSON strength | Good | Strongest — Qwen2.5 was specifically trained for structured output |

---
## Section 1 — Environment Setup

In [ ]:
import os, json, time, hashlib, warnings, random, re, copy
from pathlib import Path
from collections import Counter, defaultdict
from typing import Dict, List, Any, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

import torch
from transformers import (
    AutoModelForCausalLM, AutoTokenizer, AutoConfig,
    BitsAndBytesConfig, TrainingArguments, TrainerCallback,
)
from peft import (
    LoraConfig, get_peft_model, prepare_model_for_kbit_training,
    PeftModel, TaskType,
)
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM
from datasets import Dataset
from scipy import stats

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid", font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
plt.rcParams['savefig.dpi'] = 150

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected.")
print(f"Seed: {SEED}")

In [ ]:
# ===============================================================
# CONFIGURATION
# ===============================================================

DATASET_PATH = "auditor_dataset_rescored.jsonl"
MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"

OUTPUT_DIR = Path("training_output")
OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints_qwen"
CHECKPOINT_DIR.mkdir(exist_ok=True)
FINAL_MODEL_DIR = OUTPUT_DIR / "auditor_agent_model_qwen"
FINAL_MODEL_DIR.mkdir(exist_ok=True)
LOGS_DIR = OUTPUT_DIR / "logs_qwen"
LOGS_DIR.mkdir(exist_ok=True)

CONFIG = {
    "lora_r": 64,
    "lora_alpha": 128,
    "lora_dropout": 0.05,
    "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    "num_epochs": 5,
    "per_device_batch_size": 1,
    "gradient_accumulation": 8,
    "learning_rate": 1e-4,
    "lr_scheduler": "cosine",
    "warmup_ratio": 0.1,
    "weight_decay": 0.01,
    "max_grad_norm": 1.0,
    "max_seq_length": 8192,
    "val_ratio": 0.15,
}

for k, v in CONFIG.items():
    print(f"  {k}: {v}")

---
## Section 2 — System Prompt & Data Loading

In [ ]:
# ===============================================================
# AUDITOR SYSTEM PROMPT
# ===============================================================

AUDITOR_SYSTEM_PROMPT = """You are the strict architecture auditor.

Audit the architecture plan against:
- frozen confirmed requirements
- rich requirement notes
- cumulative issue ledger
- revision memory
- prior audit history

Main goal:
- First, verify whether previously reported issues were actually fixed.
- Second, identify the most important remaining weaknesses.
- Third, explain clearly why the score stayed the same, improved, or dropped.

Rules:
- Use stable issue IDs whenever the same issue still exists.
- Mark each issue status as one of: unresolved, resolved, downgraded, new.
- Re-check prior unresolved issues before creating new ones.
- If an earlier issue was fixed, keep the same issue ID and mark it resolved.
- If an earlier issue still exists, keep the same issue ID and explain what is still missing.
- Only create a new issue ID if the problem is materially different from previous issues.
- Score the plan against an absolute rubric, not against any approval threshold.
- Do not try to make the plan pass or fail a gate.
- Be willing to score below 9 if the plan has real weaknesses.
- If the score drops, explain the exact reason for the drop.
- If the score does not improve, explain what blocked improvement.
- Prefer the most important unresolved issues over minor nitpicks.
- passed is advisory only; the runtime decides approval.

Return JSON only with:
- thinking_summary
- rubric_scores
- summary
- strengths
- concerns
- blocking_issues
- recommendations
- requirement_conflicts
- issue_updates

rubric_scores must include numeric values from 0 to 10 for:
- requirements_alignment
- architecture_quality
- security
- operability
- internal_consistency

Each requirement_conflicts item must include:
- issue_id
- field
- current_value
- proposed_value
- exact_reason
- severity

Each issue_updates item must include:
- id
- title
- severity
- status
- detail

For each issue_updates.detail:
- State whether the issue was fixed, partially fixed, unchanged, or newly introduced.
- Explain exactly what in the plan caused this judgment.
- If the issue affected the score, explain how.
- If the architect improved one part but created another problem, say that clearly.

recommendations should:
- focus on the next highest-impact fixes
- be specific enough for the architect to act on in the next round
- avoid vague advice like \"improve architecture quality\"

summary should:
- briefly explain overall quality
- say whether the round meaningfully improved over the prior round
- mention the main reason the score changed or stayed flat"""

print(f"System prompt: {len(AUDITOR_SYSTEM_PROMPT)} characters")

In [ ]:
# ===============================================================
# LOAD DATASET
# ===============================================================

def load_jsonl(filepath):
    rows = []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError:
                    pass
    return rows

raw_data = load_jsonl(DATASET_PATH)
print(f"Loaded {len(raw_data)} rows from {DATASET_PATH}")

case_types = Counter(r.get('metadata', {}).get('case_type', '?') for r in raw_data)
pclasses = Counter(r.get('profile', {}).get('projectclass', '?') for r in raw_data)
print(f"Case types: {dict(case_types)}")
print(f"Project classes ({len(pclasses)}): {dict(pclasses)}")

---
## Section 3 — Tokenizer & Data Formatting

**Qwen2.5 chat format (ChatML):**
```
<|im_start|>system
system prompt<|im_end|>
<|im_start|>user
user message<|im_end|>
<|im_start|>assistant
assistant response<|im_end|>
```

The response template for `DataCollatorForCompletionOnlyLM` is `<|im_start|>assistant\n` —  
the tokens that appear immediately before the assistant response begins.

In [ ]:
# ===============================================================
# LOAD TOKENIZER
# ===============================================================

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print(f"Tokenizer: {MODEL_NAME}")
print(f"Vocab: {tokenizer.vocab_size}, Pad: {tokenizer.pad_token}, EOS: {tokenizer.eos_token}")

In [ ]:
# ===============================================================
# FORMAT DATASET AS TEXT WITH CHAT TEMPLATE
# ===============================================================

def get_nested(d, *keys, default=None):
    for k in keys:
        if isinstance(d, dict) and k in d:
            return d[k]
    return default

def format_row_as_text(row):
    payload = row.get('input_payload', {})
    target = row.get('target_output', {})

    auditor_input = {
        "round": payload.get('round', 1),
        "frozen_requirement_contract": get_nested(payload,
            'frozenrequirementcontract', 'frozen_requirement_contract', default={}),
        "requirements": payload.get('requirements', {}),
        "accepted_exceptions": get_nested(payload,
            'acceptedexceptions', 'accepted_exceptions', default={}),
        "issue_ledger": get_nested(payload,
            'issueledger', 'issue_ledger', default={}),
        "revision_memory": get_nested(payload,
            'revisionmemory', 'revision_memory', default={}),
        "previous_audits": get_nested(payload,
            'previousaudits', 'previous_audits', default=[]),
        "reasoner_reviews": get_nested(payload,
            'reasonerreviews', 'reasoner_reviews', default={}),
        "specialist_subplans": get_nested(payload,
            'specialistsubplans', 'specialist_subplans', default={}),
        "plan": payload.get('plan', {}),
        "best_audit": get_nested(payload,
            'bestaudit', 'best_audit', default={}),
    }

    messages = [
        {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(auditor_input, indent=1, ensure_ascii=False)},
        {"role": "assistant", "content": json.dumps(target, indent=1, ensure_ascii=False)},
    ]

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)


formatted_texts = []
chain_fps = []
sample_ids = []
skipped = 0
token_counts = []

for row in raw_data:
    try:
        text = format_row_as_text(row)
        n_tokens = len(tokenizer.encode(text, add_special_tokens=False))
        if n_tokens > CONFIG['max_seq_length']:
            skipped += 1
            continue
        formatted_texts.append(text)
        token_counts.append(n_tokens)
        sample_ids.append(row.get('sample_id', ''))

        contract = get_nested(row.get('input_payload', {}),
                    'frozenrequirementcontract', 'frozen_requirement_contract', default={})
        fp = hashlib.sha256(json.dumps(contract, sort_keys=True).encode()).hexdigest()[:16]
        chain_fps.append(fp)
    except Exception as e:
        skipped += 1

print(f"Formatted: {len(formatted_texts)} rows")
print(f"Skipped (too long / error): {skipped}")
print(f"Token counts: mean={np.mean(token_counts):.0f}, "f"median={np.median(token_counts):.0f}, max={max(token_counts)}")

# ---- CRITICAL VERIFICATION ----
print(f"\n{'='*60}")
print("FORMAT VERIFICATION")
print(f"{'='*60}")

sample = formatted_texts[0]
# Qwen2.5 ChatML: assistant response starts after <|im_start|>assistant\n
assistant_marker = "<|im_start|>assistant\n"

if assistant_marker in sample:
    pos = sample.rindex(assistant_marker) + len(assistant_marker)
    total_chars = len(sample)
    input_chars = pos
    output_chars = total_chars - pos
    print(f"  <|im_start|>assistant marker found at position {pos}")
    print(f"  Input portion:  {input_chars} chars ({input_chars/total_chars*100:.0f}%)")
    print(f"  Output portion: {output_chars} chars ({output_chars/total_chars*100:.0f}%)")

    assistant_text = sample[pos:]
    expected_keys = ["thinking_summary", "rubric_scores", "summary", "strengths", "blocking_issues", "issue_updates"]
    found = [k for k in expected_keys if k in assistant_text]
    missing = [k for k in expected_keys if k not in assistant_text]
    print(f"\n  Expected keys found in output: {found}")
    if missing:
        print(f"  WARNING - Missing keys: {missing}")
    else:
        print(f"  All expected keys present in output.")
else:
    print("  CRITICAL ERROR: <|im_start|>assistant marker not found!")
    print(f"  Sample start: {sample[:300]}")

---
## Section 4 — Completion-Only Loss Masking

**Qwen2.5 response template:** `<|im_start|>assistant\n`  
This is the ChatML marker that appears before every assistant turn.  
The collator masks everything up to and including this marker, computing loss only on the JSON output.  
Expected masking ratio: ~80-90% masked (input), ~10-20% unmasked (output).

In [ ]:
# ===============================================================
# COMPLETION-ONLY DATA COLLATOR
# ===============================================================

response_template = "<|im_start|>assistant\n"
response_template_ids = tokenizer.encode(
    response_template, add_special_tokens=False
)

print(f"Response template: '{response_template.strip()}'")
print(f"Template token IDs: {response_template_ids}")
print(f"Number of template tokens: {len(response_template_ids)}")

if len(response_template_ids) == 0:
    print("CRITICAL: response template encoded to empty — masking will fail!")

collator = DataCollatorForCompletionOnlyLM(
    response_template=response_template_ids,
    tokenizer=tokenizer,
)

# ---- VERIFY MASKING WORKS ----
print(f"\n{'='*60}")
print("LOSS MASKING VERIFICATION")
print(f"{'='*60}")

for i in range(min(5, len(formatted_texts))):
    enc = tokenizer(formatted_texts[i], return_tensors="pt", truncation=True, max_length=CONFIG['max_seq_length'])
    batch = collator([{
        "input_ids": enc["input_ids"][0],
        "attention_mask": enc["attention_mask"][0]
    }])
    labels = batch["labels"][0]
    masked = (labels == -100).sum().item()
    unmasked = (labels != -100).sum().item()
    total = len(labels)
    print(f"  Example {i}: {total} tokens, "f"masked={masked} ({masked/total*100:.0f}%), "f"unmasked={unmasked} ({unmasked/total*100:.0f}%)")

avg_unmasked_pct = []
for i in range(min(20, len(formatted_texts))):
    enc = tokenizer(formatted_texts[i], return_tensors="pt", truncation=True, max_length=CONFIG['max_seq_length'])
    batch = collator([{
        "input_ids": enc["input_ids"][0],
        "attention_mask": enc["attention_mask"][0]
    }])
    labels = batch["labels"][0]
    total = len(labels)
    unmasked = (labels != -100).sum().item()
    avg_unmasked_pct.append(unmasked / total * 100)

mean_pct = np.mean(avg_unmasked_pct)
print(f"\n  Average unmasked: {mean_pct:.1f}%")
if mean_pct < 5:
    print("  CRITICAL: Less than 5% unmasked — masking is broken! Check response_template.")
elif mean_pct > 50:
    print("  WARNING: More than 50% unmasked — masking may not be working!")
else:
    print(f"  GOOD: ~{mean_pct:.0f}% of tokens are output (loss computed on these)")

---
## Section 5 — Chain-Aware Train/Validation Split

Identical split logic to Phi-3 run — same seed, same ratio — for fair comparison.

In [ ]:
# ===============================================================
# CHAIN-AWARE SPLIT
# ===============================================================

chains = defaultdict(list)
for i, fp in enumerate(chain_fps):
    chains[fp].append(i)

chain_ids = list(chains.keys())
rng = random.Random(SEED)
rng.shuffle(chain_ids)

n_val = max(1, int(len(chain_ids) * CONFIG['val_ratio']))
val_chain_set = set(chain_ids[:n_val])

train_idx, val_idx = [], []
for cid in chain_ids:
    target = val_idx if cid in val_chain_set else train_idx
    target.extend(chains[cid])

train_texts = [formatted_texts[i] for i in train_idx]
val_texts = [formatted_texts[i] for i in val_idx]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset = Dataset.from_dict({"text": val_texts})

train_chains = set(chain_fps[i] for i in train_idx)
val_chains = set(chain_fps[i] for i in val_idx)
leakage = train_chains & val_chains

print(f"Chains: {len(chains)} total, {len(chains) - n_val} train, {n_val} val")
print(f"Rows: {len(train_texts)} train, {len(val_texts)} val")
print(f"Chain leakage: {'NONE (good)' if not leakage else f'DETECTED: {len(leakage)} chains!'}")

---
## Section 6 — Model Loading with 4-bit Quantisation

NF4 quantisation reduces the 7B Qwen2.5 model from ~14GB to ~4GB.  
**Qwen2.5 differences from Phi-3:** `trust_remote_code=False`, no `pretraining_tp`, no HF token.

In [ ]:
# ===============================================================
# QUANTIZATION CONFIG
# ===============================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

config = AutoConfig.from_pretrained(MODEL_NAME, trust_remote_code=False)

print(f"Loading {MODEL_NAME}...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    config=config,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=False,
    attn_implementation="eager",
    torch_dtype=torch.bfloat16,
)

model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total_params:,} parameters")
print(f"GPU memory: {torch.cuda.memory_allocated()/1e9:.2f} GB")

---
## Section 7 — LoRA Adapter Configuration

Identical LoRA config to Phi-3 run for fair comparison.

In [ ]:
# ===============================================================
# APPLY LoRA
# ===============================================================

lora_config = LoraConfig(
    r=CONFIG["lora_r"],
    lora_alpha=CONFIG["lora_alpha"],
    lora_dropout=CONFIG["lora_dropout"],
    target_modules=CONFIG["target_modules"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
pct = trainable / total * 100

print(f"Trainable: {trainable:,} / {total:,} ({pct:.2f}%)")
model.print_trainable_parameters()

---
## Section 8 — Training Configuration & Execution

All hyperparameters identical to Phi-3 run for fair comparison.

In [ ]:
# ===============================================================
# TRAINING ARGS
# ===============================================================

total_steps = len(train_dataset) * CONFIG['num_epochs'] // CONFIG['gradient_accumulation']
eval_steps = max(1, int(total_steps * 0.1))
save_steps = eval_steps * 3

print(f"Training plan:")
print(f"  Examples: {len(train_dataset)} train, {len(val_dataset)} val")
print(f"  Epochs: {CONFIG['num_epochs']}, Effective batch: {CONFIG['gradient_accumulation']}")
print(f"  Steps: ~{total_steps}, Eval every {eval_steps}, Save every {save_steps}")

training_args = TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=CONFIG['num_epochs'],
    per_device_train_batch_size=CONFIG['per_device_batch_size'],
    per_device_eval_batch_size=CONFIG['per_device_batch_size'],
    gradient_accumulation_steps=CONFIG['gradient_accumulation'],
    learning_rate=CONFIG['learning_rate'],
    lr_scheduler_type=CONFIG['lr_scheduler'],
    warmup_ratio=CONFIG['warmup_ratio'],
    weight_decay=CONFIG['weight_decay'],
    max_grad_norm=CONFIG['max_grad_norm'],
    fp16=False,
    bf16=True,
    logging_dir=str(LOGS_DIR),
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=eval_steps,
    save_strategy="steps",
    save_steps=save_steps,
    save_total_limit=3,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    report_to="tensorboard",
    seed=SEED,
)
print("Training args ready.")

In [ ]:
# ===============================================================
# METRICS CALLBACK
# ===============================================================

class MetricsCallback(TrainerCallback):
    def __init__(self):
        self.train_losses, self.eval_losses = [], []
        self.learning_rates, self.steps, self.eval_steps_list = [], [], []

    def on_log(self, args, state, control, logs=None, **kwargs):
        if logs:
            if 'loss' in logs:
                self.train_losses.append(logs['loss'])
                self.steps.append(state.global_step)
            if 'learning_rate' in logs:
                self.learning_rates.append(logs['learning_rate'])
            if 'eval_loss' in logs:
                self.eval_losses.append(logs['eval_loss'])
                self.eval_steps_list.append(state.global_step)

metrics_cb = MetricsCallback()

In [ ]:
# ===============================================================
# TRAIN WITH COMPLETION-ONLY LOSS
# ===============================================================

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=collator,
    max_seq_length=CONFIG['max_seq_length'],
    dataset_text_field="text",
    packing=False,
    callbacks=[metrics_cb],
)

print("Starting training with completion-only loss masking...")
print("=" * 60)

start_time = time.time()
train_result = trainer.train()
elapsed = time.time() - start_time

print("=" * 60)
print(f"TRAINING COMPLETE")
print(f"  Duration: {elapsed/60:.1f} minutes")
print(f"  Final loss: {train_result.training_loss:.4f}")
print(f"  Steps: {train_result.global_step}")

trainer.save_metrics("train", train_result.metrics)

In [ ]:
# ===============================================================
# EVALUATE
# ===============================================================

trainer.callback_handler.callbacks = [
    cb for cb in trainer.callback_handler.callbacks
    if "NotebookProgress" not in type(cb).__name__
]

eval_results = trainer.evaluate()
eval_loss = eval_results.get('eval_loss', 0)
perplexity = np.exp(eval_loss) if eval_loss < 20 else float('inf')

print(f"Eval loss: {eval_loss:.4f}")
print(f"Perplexity: {perplexity:.2f}")
trainer.save_metrics("eval", eval_results)

---
## Section 9 — Save Model

In [ ]:
# ===============================================================
# SAVE LoRA ADAPTER
# ===============================================================

adapter_dir = FINAL_MODEL_DIR / "lora_adapter"
adapter_dir.mkdir(exist_ok=True)

model.save_pretrained(str(adapter_dir))
tokenizer.save_pretrained(str(adapter_dir))

adapter_size = sum(f.stat().st_size for f in adapter_dir.rglob('*') if f.is_file()) / 1e6
print(f"Adapter saved: {adapter_dir} ({adapter_size:.1f} MB)")

---
## Section 10 — Quick Generation Test

**Qwen2.5 end token:** `<|im_end|>` — different from Phi-3's `<|end|>` and Mistral's `</s>`.  
Using the wrong end token causes the model to keep generating past the closing brace.

In [ ]:
# ===============================================================
# QUICK GENERATION TEST
# ===============================================================

model.eval()

test_row = raw_data[0]
payload = test_row.get('input_payload', {})

test_input = {
    "round": payload.get('round', 1),
    "frozen_requirement_contract": get_nested(payload,
        'frozenrequirementcontract', 'frozen_requirement_contract', default={}),
    "requirements": payload.get('requirements', {}),
    "accepted_exceptions": get_nested(payload,
        'acceptedexceptions', 'accepted_exceptions', default={}),
    "issue_ledger": get_nested(payload,
        'issueledger', 'issue_ledger', default={}),
    "revision_memory": get_nested(payload,
        'revisionmemory', 'revision_memory', default={}),
    "previous_audits": get_nested(payload,
        'previousaudits', 'previous_audits', default=[]),
    "plan": payload.get('plan', {}),
}

messages = [
    {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
    {"role": "user", "content": json.dumps(test_input, indent=1, ensure_ascii=False)},
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                    max_length=CONFIG['max_seq_length'] - 2000)
inputs = {k: v.to(model.device) for k, v in inputs.items()}

end_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")

print(f"Input tokens: {inputs['input_ids'].shape[1]}")
print(f"End token: <|im_end|> = {end_token_id}")
print("Generating...")

with torch.no_grad():
    out = model.generate(
        **inputs,
        max_new_tokens=2000,
        do_sample=False,
        temperature=1.0,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=end_token_id,
        use_cache=True,
    )

gen_ids = out[0][inputs['input_ids'].shape[1]:]
generated = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()

if generated.startswith("```"):
    generated = re.sub(r'^```[a-zA-Z]*\n?', '', generated)
    generated = re.sub(r'\n?```$', '', generated).strip()

print(f"\nGenerated ({len(generated)} chars):")
print(generated)

try:
    start = generated.find('{')
    end = generated.rfind('}')
    if start != -1 and end > start:
        parsed = json.loads(generated[start:end+1])
        print(f"\n{'='*60}")
        print(f"JSON VALID: YES")
        print(f"Keys: {list(parsed.keys())}")
        rubric = parsed.get('rubric_scores', {})
        print(f"Rubric scores: {rubric}")
        issues = parsed.get('issue_updates', [])
        print(f"Issues: {len(issues)}")
        if issues:
            print(f"First issue: {issues[0].get('id','')} - {issues[0].get('title','')}")
    else:
        print(f"\nJSON VALID: NO (no JSON object found)")
except Exception as e:
    print(f"\nJSON VALID: NO ({e})")

---
## Section 11 — Full Structural & Semantic Evaluation

Generate audit outputs for 50 validation examples and measure:
1. **Structural metrics**: JSON validity, schema compliance, rubric ranges, issue template, enum compliance
2. **Semantic metrics**: Rubric score MAE vs reference, issue detection F1, blocking issue agreement
3. **ROUGE-L**: Text similarity between generated and reference text fields
4. **LLM-as-a-Judge**: GPT-5-Chat scores the generated audits against reference on 5 dimensions

Same metrics as all other runs — enables direct comparison.

In [ ]:
# ===============================================================
# GENERATE ON VALIDATION SET
# ===============================================================

def generate_audit(model, tokenizer, row, max_new_tokens=2000):
    payload = row.get('input_payload', {})
    auditor_input = {
        "round": payload.get('round', 1),
        "frozen_requirement_contract": get_nested(payload,
            'frozenrequirementcontract', 'frozen_requirement_contract', default={}),
        "requirements": payload.get('requirements', {}),
        "accepted_exceptions": get_nested(payload,
            'acceptedexceptions', 'accepted_exceptions', default={}),
        "issue_ledger": get_nested(payload,
            'issueledger', 'issue_ledger', default={}),
        "revision_memory": get_nested(payload,
            'revisionmemory', 'revision_memory', default={}),
        "previous_audits": get_nested(payload,
            'previousaudits', 'previous_audits', default=[]),
        "reasoner_reviews": get_nested(payload,
            'reasonerreviews', 'reasoner_reviews', default={}),
        "specialist_subplans": get_nested(payload,
            'specialistsubplans', 'specialist_subplans', default={}),
        "plan": payload.get('plan', {}),
        "best_audit": get_nested(payload,
            'bestaudit', 'best_audit', default={}),
    }

    messages = [
        {"role": "system", "content": AUDITOR_SYSTEM_PROMPT},
        {"role": "user", "content": json.dumps(auditor_input, indent=1, ensure_ascii=False)},
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True,
                        max_length=CONFIG['max_seq_length'] - max_new_tokens)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    end_token_id = tokenizer.convert_tokens_to_ids("<|im_end|>")  # <-- CHANGED

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=end_token_id,
            use_cache=True,
        )

    gen_ids = outputs[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(gen_ids, skip_special_tokens=True).strip()


def extract_json(text):
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r'^```[a-zA-Z]*\n?', '', text)
        text = re.sub(r'\n?```$', '', text).strip()
    try:
        return json.loads(text)
    except:
        pass
    start = text.find('{')
    if start == -1:
        return None
    depth = 0
    in_string = False
    escape_next = False
    for i, ch in enumerate(text[start:], start):
        if escape_next:
            escape_next = False
            continue
        if ch == '\\' and in_string:
            escape_next = True
            continue
        if ch == '"':
            in_string = not in_string
            continue
        if in_string:
            continue
        if ch == '{':
            depth += 1
        elif ch == '}':
            depth -= 1
            if depth == 0:
                try:
                    return json.loads(text[start:i+1])
                except:
                    return None
    return None


val_sample_ids = set(sample_ids[i] for i in val_idx)
val_rows = [r for r in raw_data if r.get('sample_id', '') in val_sample_ids]

EVAL_SIZE = min(50, len(val_rows))
eval_sample = random.Random(SEED).sample(val_rows, EVAL_SIZE)

print(f"Generating on {EVAL_SIZE} validation examples...")

predictions = []
start_global = time.time()

f = open(str(OUTPUT_DIR / "validation_outputs_qwen.jsonl"), "w", encoding="utf-8")

for i, row in enumerate(eval_sample):
    sid = row.get('sample_id', f'val_{i}')
    start_sample = time.time()

    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{EVAL_SIZE}] {sid}")

    try:
        gen_text = generate_audit(model, tokenizer, row)
        parsed = extract_json(gen_text)
    except Exception as e:
        print(f"  ERROR {sid}: {str(e)[:80]}")
        gen_text = ""
        parsed = None

    predictions.append({
        "sample_id": sid,
        "raw": gen_text[:600],
        "parsed": parsed,
        "reference": row.get('target_output', {}),
        "json_valid": parsed is not None,
    })
    f.write(json.dumps(predictions[-1], ensure_ascii=False) + "\n")

    elapsed_sample = time.time() - start_sample
    elapsed_total = time.time() - start_global
    avg_time = elapsed_total / (i + 1)
    remaining = avg_time * (EVAL_SIZE - (i + 1))
    print(f"[{i+1}/{EVAL_SIZE}] {elapsed_sample:.1f}s | ETA: {remaining/60:.1f}m")

    if i < 3:
        print(f"\n  --- {sid} ---")
        print(f"  JSON valid: {parsed is not None}")
        if parsed:
            print(f"  Keys: {list(parsed.keys())[:6]}")
            print(f"  Rubric: {parsed.get('rubric_scores', {})}")
        else:
            print(f"  Raw: {gen_text[:200]}")
        print()

f.close()
valid = sum(1 for p in predictions if p['json_valid'])
print(f"\nDone. JSON valid: {valid}/{EVAL_SIZE} ({valid/EVAL_SIZE*100:.1f}%)")

In [ ]:
# ===============================================================
# COMPUTE ALL METRICS
# ===============================================================

REQUIRED_KEYS = {"thinking_summary", "rubric_scores", "summary", "strengths", "concerns", "blocking_issues", "recommendations", "requirement_conflicts", "issue_updates"}
RUBRIC_DIMS = ["requirements_alignment", "architecture_quality", "security", "operability", "internal_consistency"]
ALLOWED_SEV = {"critical", "high", "medium", "low"}
ALLOWED_STAT = {"unresolved", "resolved", "downgraded", "new"}
DETAIL_RE = re.compile(r"^(Fixed|Partially fixed|Unchanged|Newly introduced)\.?", re.IGNORECASE)

n = len(predictions)
json_valid = sum(1 for p in predictions if p['parsed'] is not None)
schema_ok = 0
rubric_ok = 0
template_ok = 0
template_total = 0
enum_ok = 0
enum_total = 0

rubric_errors = {d: [] for d in RUBRIC_DIMS}
issue_f1s = []
issue_f1s_category = []
blocking_matches = []

for pred in predictions:
    parsed = pred['parsed']
    ref = pred['reference']
    if not parsed:
        continue

    present_keys = set(parsed.keys())
    if REQUIRED_KEYS.issubset(present_keys):
        schema_ok += 1

    rubric = parsed.get('rubric_scores', {})
    if isinstance(rubric, dict):
        all_valid = True
        for d in RUBRIC_DIMS:
            val = rubric.get(d)
            if isinstance(val, (int, float)) and 0 <= val <= 10:
                ref_rubric = ref.get('rubric_scores', ref.get('rubricscores', {}))
                ref_val = ref_rubric.get(d, ref_rubric.get(d.replace('_', ''), None))
                if isinstance(ref_val, (int, float)):
                    rubric_errors[d].append(abs(val - ref_val))
            else:
                all_valid = False
        if all_valid and sum(1 for d in RUBRIC_DIMS if d in rubric) == 5:
            rubric_ok += 1

    issues = parsed.get('issue_updates', [])
    if isinstance(issues, list):
        for iu in issues:
            if not isinstance(iu, dict):
                continue
            sev = str(iu.get('severity', '')).lower()
            stat = str(iu.get('status', '')).lower()
            enum_total += 1
            if sev in ALLOWED_SEV and stat in ALLOWED_STAT:
                enum_ok += 1
            detail = str(iu.get('detail', ''))
            template_total += 1
            if DETAIL_RE.match(detail):
                template_ok += 1

    pred_ids = {str(iu.get('id', '')) for iu in issues if isinstance(iu, dict)}
    ref_issues_list = ref.get('issue_updates', ref.get('issueupdates', []))
    ref_ids = {str(iu.get('id', '')) for iu in (ref_issues_list or []) if isinstance(iu, dict)}
    if ref_ids:
        tp = len(pred_ids & ref_ids)
        prec = tp / max(len(pred_ids), 1)
        rec = tp / max(len(ref_ids), 1)
        f1 = 2 * prec * rec / max(prec + rec, 1e-8)
        issue_f1s.append(f1)

    def get_category(issue_id):
        parts = str(issue_id).split('-')
        return parts[0].upper() if parts else 'UNK'

    pred_cats = {get_category(iu.get('id', '')) for iu in issues if isinstance(iu, dict)}
    ref_cats = {get_category(iu.get('id', '')) for iu in (ref_issues_list or []) if isinstance(iu, dict)}
    if ref_cats:
        tp_cat = len(pred_cats & ref_cats)
        prec_cat = tp_cat / max(len(pred_cats), 1)
        rec_cat = tp_cat / max(len(ref_cats), 1)
        f1_cat = 2 * prec_cat * rec_cat / max(prec_cat + rec_cat, 1e-8)
        issue_f1s_category.append(f1_cat)

    pb = len(parsed.get('blocking_issues', []) or [])
    rb = len(ref.get('blocking_issues', ref.get('blockingissues', [])) or [])
    blocking_matches.append((pb > 0) == (rb > 0))

structural = {
    "json_validity_rate": json_valid / n,
    "schema_compliance_rate": schema_ok / n,
    "rubric_range_compliance": rubric_ok / n,
    "issue_template_compliance": template_ok / max(template_total, 1),
    "severity_status_compliance": enum_ok / max(enum_total, 1),
}

print(f"\n{'='*60}")
print(f"STRUCTURAL METRICS (n={n})")
print(f"{'='*60}")
for k, v in structural.items():
    bar = '#' * int(v * 30)
    print(f"  {k:35s}: {v:6.1%}  {bar}")

print(f"\n{'='*60}")
print(f"SEMANTIC METRICS")
print(f"{'='*60}")
overall_mae = []
for d in RUBRIC_DIMS:
    if rubric_errors[d]:
        mae = np.mean(rubric_errors[d])
        overall_mae.extend(rubric_errors[d])
        print(f"  {d:30s} MAE: {mae:.2f}")
    else:
        print(f"  {d:30s} MAE: no data")

if overall_mae:
    print(f"  {'OVERALL':30s} MAE: {np.mean(overall_mae):.2f}")
if issue_f1s:
    print(f"  Issue Detection F1: {np.mean(issue_f1s):.2%}")
if blocking_matches:
    print(f"  Blocking Agreement: {np.mean(blocking_matches):.2%}")

all_metrics = {
    "training": {
        "train_loss": train_result.training_loss,
        "eval_loss": eval_loss,
        "perplexity": perplexity,
        "duration_minutes": elapsed / 60,
        "trainable_params": trainable,
        "trainable_pct": pct,
    },
    "structural": structural,
    "semantic": {
        "rubric_mae": {d: float(np.mean(rubric_errors[d])) if rubric_errors[d] else None for d in RUBRIC_DIMS},
        "rubric_mae_overall": float(np.mean(overall_mae)) if overall_mae else None,
        "issue_f1": float(np.mean(issue_f1s)) if issue_f1s else None,
        "issue_f1_category": float(np.mean(issue_f1s_category)) if issue_f1s_category else None,
        "blocking_agreement": float(np.mean(blocking_matches)) if blocking_matches else None,
    }
}
with open(OUTPUT_DIR / 'all_metrics_qwen.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f"\nAll metrics saved to {OUTPUT_DIR / 'all_metrics_qwen.json'}")

In [ ]:
# ===============================================================
# ROUGE-L — TEXT FIELD SIMILARITY
# ===============================================================

from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
text_fields = ['thinking_summary', 'summary']
rouge_results = {field: [] for field in text_fields}

for pred in predictions:
    parsed = pred['parsed']
    ref = pred['reference']
    if not parsed:
        continue
    for field in text_fields:
        pred_text = str(parsed.get(field, '') or '')
        ref_text = str(ref.get(field, '') or '')
        if pred_text and ref_text:
            score = scorer.score(ref_text, pred_text)
            rouge_results[field].append(score['rougeL'].fmeasure)

print('\nROUGE-L SCORES (text field similarity)')
print('='*60)
for field in text_fields:
    if rouge_results[field]:
        print(f'  {field:30s}: {np.mean(rouge_results[field]):.3f}')

rouge_metrics = {
    field: float(np.mean(rouge_results[field])) if rouge_results[field] else None
    for field in text_fields
}
all_metrics['rouge_l'] = rouge_metrics

with open(OUTPUT_DIR / 'all_metrics_qwen.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
print(f'\nROUGE-L saved to {OUTPUT_DIR / "all_metrics_qwen.json"}')

In [ ]:
# ===============================================================
# LLM-AS-A-JUDGE
# ===============================================================

import openai

client = openai.AzureOpenAI(
    api_key="F79rr24XOyTKAprSSVMiQuo8j99MQM9gzJD3oEIAmlfn4vrsj0TVJQQJ99CBACHYHv6XJ3w3AAABACOGX5Md",
    api_version="2024-08-01-preview",
    azure_endpoint="https://cmg-ai-poc-eu2.openai.azure.com/"
)
JUDGE_MODEL = "gpt-5-chat"

JUDGE_PROMPT = """You are an expert software architecture reviewer evaluating an AI-generated audit report.

You will be given:
1. The REFERENCE audit — written as the ground truth
2. The PREDICTED audit — generated by a fine-tuned language model

Score the predicted audit on these 5 dimensions, each from 0 to 10:

1. **issue_identification** (0-10): Did the predicted audit identify the same architectural issues as the reference? Award points for correctly flagging problems even if the wording differs.

2. **reasoning_quality** (0-10): Is the thinking_summary and detail reasoning coherent, specific, and grounded in the actual plan content? Penalise vague or generic statements.

3. **recommendation_actionability** (0-10): Are the recommendations specific enough for an architect to act on in the next round? Penalise advice like "improve security" with no specifics.

4. **rubric_calibration** (0-10): How well do the predicted rubric scores (0-10 per dimension) align with the reference scores in terms of relative ordering and magnitude?

5. **overall_audit_quality** (0-10): Overall, how useful would this audit be to an architect trying to improve their plan?

Return ONLY a JSON object with no extra text:
{
    "issue_identification": <int 0-10>,
    "reasoning_quality": <int 0-10>,
    "recommendation_actionability": <int 0-10>,
    "rubric_calibration": <int 0-10>,
    "overall_audit_quality": <int 0-10>,
    "brief_justification": "<2-3 sentences explaining the scores>"
}"""


def judge_single(reference, predicted, client, model):
    user_content = f"""REFERENCE AUDIT:
{json.dumps(reference, indent=2, ensure_ascii=False)}

PREDICTED AUDIT:
{json.dumps(predicted, indent=2, ensure_ascii=False)}"""
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[
                {"role": "system", "content": JUDGE_PROMPT},
                {"role": "user", "content": user_content}
            ],
            temperature=0.0,
            max_tokens=400,
        )
        raw = response.choices[0].message.content.strip()
        if raw.startswith("```"):
            raw = re.sub(r'^```[a-zA-Z]*\n?', '', raw)
            raw = re.sub(r'\n?```$', '', raw).strip()
        return json.loads(raw)
    except Exception as e:
        return {"error": str(e)}


JUDGE_DIMS = ["issue_identification", "reasoning_quality", "recommendation_actionability", "rubric_calibration", "overall_audit_quality"]

judge_scores = {d: [] for d in JUDGE_DIMS}
judge_results = []
failed = 0

print(f"Running LLM judge on {len(predictions)} predictions...")

for i, pred in enumerate(predictions):
    if not pred['parsed']:
        failed += 1
        continue

    scores = judge_single(pred['reference'], pred['parsed'], client, JUDGE_MODEL)

    if 'error' in scores:
        print(f"  [{i+1}] ERROR: {scores['error'][:60]}")
        failed += 1
        continue

    for dim in JUDGE_DIMS:
        if isinstance(scores.get(dim), (int, float)):
            judge_scores[dim].append(float(scores[dim]))

    judge_results.append({"sample_id": pred['sample_id'], "scores": scores})

    if (i + 1) % 10 == 0:
        print(f"  [{i+1}/{len(predictions)}] avg overall: {np.mean(judge_scores['overall_audit_quality']):.2f}")

    time.sleep(0.5)

print(f"\n{'='*60}")
print(f"LLM JUDGE RESULTS — Qwen2.5-7B (n={len(judge_results)})")
print(f"{'='*60}")
for dim in JUDGE_DIMS:
    if judge_scores[dim]:
        mean = np.mean(judge_scores[dim])
        bar = '#' * int(mean * 3)
        print(f"  {dim:35s}: {mean:.2f}/10  {bar}")

print(f"\n  failed/skipped: {failed}")

all_metrics['llm_judge'] = {
    dim: float(np.mean(judge_scores[dim])) if judge_scores[dim] else None
    for dim in JUDGE_DIMS
}

with open(OUTPUT_DIR / 'all_metrics_qwen.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)

with open(OUTPUT_DIR / 'llm_judge_results_qwen.json', 'w', encoding='utf-8') as f:
    json.dump(judge_results, f, indent=2, ensure_ascii=False)

print(f"\nSaved: {OUTPUT_DIR / 'all_metrics_qwen.json'}")
print(f"Saved: {OUTPUT_DIR / 'llm_judge_results_qwen.json'}")

---
## Section 12 — Results Visualisation

In [ ]:
# ===============================================================
# TRAINING CURVES
# ===============================================================

if metrics_cb.train_losses:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle('Qwen2.5-7B-Instruct — AuditorAgent Training', fontsize=14, fontweight='bold')

    ax = axes[0]
    ax.plot(metrics_cb.steps, metrics_cb.train_losses, alpha=0.4, color='#4C72B0', linewidth=0.8, label='train loss (raw)')
    if len(metrics_cb.train_losses) > 5:
        window = max(1, len(metrics_cb.train_losses) // 15)
        smoothed = np.convolve(metrics_cb.train_losses, np.ones(window)/window, mode='valid')
        ax.plot(metrics_cb.steps[window-1:], smoothed, color='#4C72B0', linewidth=2, label='train loss (smoothed)')
    if metrics_cb.eval_steps_list:
        ax.plot(metrics_cb.eval_steps_list, metrics_cb.eval_losses, 'o-', color='#DD8452', linewidth=2, markersize=5, label='eval loss')
    ax.set_xlabel('Step')
    ax.set_ylabel('Loss')
    ax.set_title('Training and Validation Loss')
    ax.legend()
    ax.grid(True, alpha=0.3)

    ax2 = axes[1]
    if metrics_cb.learning_rates:
        ax2.plot(metrics_cb.steps[:len(metrics_cb.learning_rates)], metrics_cb.learning_rates, color='#59A14F', linewidth=1.5)
    ax2.set_xlabel('Step')
    ax2.set_ylabel('Learning Rate')
    ax2.set_title('Cosine LR Schedule')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / 'training_curves_qwen.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved: training_curves_qwen.png")

In [ ]:
# ===============================================================
# EVALUATION VISUALISATION
# ===============================================================

fig = plt.figure(figsize=(16, 10))
fig.suptitle('Qwen2.5-7B-Instruct — AuditorAgent Evaluation Results', fontsize=14, fontweight='bold')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

ax1 = fig.add_subplot(gs[0, 0])
names = ['JSON\nValid', 'Schema', 'Rubric\nRange', 'Issue\nTemplate', 'Sev/Status\nEnum']
vals = list(structural.values())
colors = ['#55A868' if v > 0.7 else '#DD8452' if v > 0.4 else '#C44E52' for v in vals]
bars = ax1.bar(names, vals, color=colors, edgecolor='black', alpha=0.8)
ax1.set_ylim(0, 1.05)
ax1.set_ylabel('Compliance Rate')
ax1.set_title('Structural Output Quality')
for bar, val in zip(bars, vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02, f'{val:.0%}', ha='center', fontsize=9)

ax2 = fig.add_subplot(gs[0, 1])
dim_short = ['ReqAlign', 'ArchQual', 'Security', 'Operab', 'IntConsist']
dim_maes = [np.mean(rubric_errors[d]) if rubric_errors[d] else 0 for d in RUBRIC_DIMS]
mae_colors = ['#55A868' if m < 1.5 else '#DD8452' if m < 2.5 else '#C44E52' for m in dim_maes]
bars2 = ax2.bar(dim_short, dim_maes, color=mae_colors, edgecolor='black', alpha=0.8)
ax2.set_ylabel('Mean Absolute Error')
ax2.set_title('Rubric Score MAE')
ax2.axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Good (<1.0)')
ax2.axhline(2.0, color='orange', linestyle='--', alpha=0.5, label='Acceptable (<2.0)')
ax2.legend(fontsize=7)
for bar, val in zip(bars2, dim_maes):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, f'{val:.2f}', ha='center', fontsize=9)

ax3 = fig.add_subplot(gs[0, 2])
pred_scores, ref_scores = [], []
for pred in predictions:
    if pred['parsed'] and pred['reference']:
        p_rub = pred['parsed'].get('rubric_scores', {})
        r_rub = pred['reference'].get('rubric_scores', pred['reference'].get('rubricscores', {}))
        for d in RUBRIC_DIMS:
            pv = p_rub.get(d)
            rv = r_rub.get(d, r_rub.get(d.replace('_', ''), None))
            if isinstance(pv, (int, float)) and isinstance(rv, (int, float)):
                pred_scores.append(pv)
                ref_scores.append(rv)

if pred_scores:
    ax3.scatter(ref_scores, pred_scores, alpha=0.3, s=20, c='#8172B3')
    ax3.plot([0, 10], [0, 10], 'r--', alpha=0.5, label='Perfect')
    ax3.set_xlabel('Reference Score')
    ax3.set_ylabel('Predicted Score')
    ax3.set_title('Predicted vs Reference Rubric')
    ax3.set_xlim(-0.5, 10.5)
    ax3.set_ylim(-0.5, 10.5)
    if len(pred_scores) > 2:
        r_val, _ = stats.pearsonr(ref_scores, pred_scores)
        ax3.annotate(f'r={r_val:.3f}', xy=(0.05, 0.92), xycoords='axes fraction')
    ax3.legend()

ax4 = fig.add_subplot(gs[1, 0])
if issue_f1s:
    ax4.hist(issue_f1s, bins=10, edgecolor='black', alpha=0.7, color='#4C72B0')
    ax4.axvline(np.mean(issue_f1s), color='red', linestyle='--',
                label=f'Mean={np.mean(issue_f1s):.2f}')
    ax4.set_xlabel('F1 Score')
    ax4.set_ylabel('Count')
    ax4.set_title('Issue Detection F1')
    ax4.legend()

ax5 = fig.add_subplot(gs[1, 1:])
ax5.axis('off')
summary_rows = [
    ['Model', 'Qwen2.5-7B-Instruct'],
    ['Training Loss', f'{train_result.training_loss:.4f}'],
    ['Eval Loss', f'{eval_loss:.4f}'],
    ['Perplexity', f'{perplexity:.2f}'],
    ['Training Time', f'{elapsed/60:.1f} min'],
    ['Trainable Params', f'{trainable:,} ({pct:.2f}%)'],
    ['JSON Validity', f'{structural["json_validity_rate"]:.1%}'],
    ['Schema Compliance', f'{structural["schema_compliance_rate"]:.1%}'],
    ['Rubric MAE', f'{np.mean(overall_mae):.2f}' if overall_mae else 'N/A'],
    ['Issue Detection F1', f'{np.mean(issue_f1s):.2%}' if issue_f1s else 'N/A'],
    ['Blocking Agreement', f'{np.mean(blocking_matches):.2%}' if blocking_matches else 'N/A'],
]
table = ax5.table(cellText=summary_rows, colLabels=['Metric', 'Value'], loc='center', cellLoc='left')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
ax5.set_title('Comprehensive Results Summary', fontsize=13, fontweight='bold', pad=20)

plt.savefig(OUTPUT_DIR / 'evaluation_results_qwen.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: evaluation_results_qwen.png")

---
## Section 13 — Save Training Record

In [ ]:
# ===============================================================
# SAVE TRAINING RECORD
# ===============================================================

SAVE_MERGED = True

if SAVE_MERGED:
    print("Merging adapter into base model...")
    merged_dir = FINAL_MODEL_DIR / "merged_model"
    merged_dir.mkdir(exist_ok=True)
    merged_model = model.merge_and_unload()
    merged_model.save_pretrained(str(merged_dir), safe_serialization=True, max_shard_size="2GB")
    tokenizer.save_pretrained(str(merged_dir))
    merged_size = sum(f.stat().st_size for f in merged_dir.rglob('*') if f.is_file()) / 1e9
    print(f"Merged model saved: {merged_dir} ({merged_size:.2f} GB)")
else:
    print("Skipped merged model (SAVE_MERGED=False)")

record = {
    "model_name": MODEL_NAME,
    "dataset_path": DATASET_PATH,
    "dataset_size": len(raw_data),
    "formatted_size": len(formatted_texts),
    "train_size": len(train_dataset),
    "val_size": len(val_dataset),
    "config": CONFIG,
    "trainable_parameters": trainable,
    "trainable_percentage": pct,
    "training_loss": train_result.training_loss,
    "eval_loss": eval_loss,
    "perplexity": perplexity,
    "training_duration_minutes": elapsed / 60,
    "structural_metrics": structural,
    "semantic_metrics": all_metrics.get("semantic", {}),
    "completion_only_masking": True,
    "response_template": "<|im_start|>assistant\n",
    "end_token": "<|im_end|>",
    "seed": SEED,
}

with open(FINAL_MODEL_DIR / 'training_record.json', 'w') as f:
    json.dump(record, f, indent=2, default=str)

print(f"Training record saved to {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"\n{'='*60}")
print("ALL OUTPUTS:")
print(f"  Adapter:    {FINAL_MODEL_DIR / 'lora_adapter'}")
print(f"  Curves:     {OUTPUT_DIR / 'training_curves_qwen.png'}")
print(f"  Evaluation: {OUTPUT_DIR / 'evaluation_results_qwen.png'}")
print(f"  Metrics:    {OUTPUT_DIR / 'all_metrics_qwen.json'}")
print(f"  Judge:      {OUTPUT_DIR / 'llm_judge_results_qwen.json'}")
print(f"  Record:     {FINAL_MODEL_DIR / 'training_record.json'}")
print(f"{'='*60}")

In [ ]:
import shutil
shutil.make_archive('/workspace/training_output_qwen', 'zip', '/workspace/training_output')